# Concept measurement reliability

Frozen activation **A** and original observable rule **R** remain separate. Reference patients select one CAV procedure; a private freeze precedes exploratory temporal evaluation. Estimates, support and uncertainty are reported without concept-quality labels.

In [1]:
import os, time, warnings, pickle
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score
from cmr_common import setup_run, iter_units, freeze_write, freeze_read, publish, patient_weights, finish_run, check_budget

MODE = os.environ.get("CMR_MODE", "check")
C_GRID = (.01, .1, 1.)
METHODS = ("legacy", "all_negatives")
JOINT_SEEDS = tuple(range(1001, 1051))
NEGATIVE_SEEDS = (42, 43, 44)
LABEL_SEEDS = tuple(range(2001, 2006))
INTERVAL_SEEDS = tuple(range(3001, 3501))
OUTCOME_LAST_YEAR, TIE, MIN_CLASS = 2014, 1e-6, 2
SMOKE_SECONDS, TOTAL_SECONDS = 1200, 8 * 3600
ctx = setup_run("measurement") if MODE != "check" else None
print(f"Measurement mode: {MODE}; 50 fitting draws; 500 conditional patient draws")

Measurement mode: evaluate; 50 fitting draws; 500 conditional patient draws


## Independent support and fixed contrast

CAV positives satisfy R. Eligible negatives fail R and lie at the minimum finite reference CAV-fitting activation. This cutoff is unrelated to the saved A cutoff. Multiplicity weights preserve each class's fixed reference mass.

In [2]:
def class_weights(labels, multiplicity, mass):
    labels, multiplicity = np.asarray(labels), np.asarray(multiplicity, float)
    weights = np.zeros(len(labels), float)
    for label in (0, 1):
        at = labels == label
        total = multiplicity[at].sum()
        if total:
            weights[at] = multiplicity[at] * (mass / 2) / total
    return weights

def contrast_rows(unit, rows, negative_cutoff):
    rows = np.asarray(rows, int)
    rule, activation = unit["rule_mask"][rows], unit["activation"][rows]
    return rows[rule | (~rule & np.isfinite(activation) & (activation <= negative_cutoff))]

def contrast_support(unit):
    at = unit["cav_fit"]
    finite = unit["activation"][at][np.isfinite(unit["activation"][at])]
    cutoff = float(finite.min()) if len(finite) else np.nan
    fit = contrast_rows(unit, at, cutoff)
    selection = contrast_rows(unit, unit["selection"], cutoff)
    counts = {}
    for name, rows in (("fit", fit), ("selection", selection)):
        for label in (0, 1):
            ids = unit["patients"][rows][unit["rule_mask"][rows] == label]
            counts[f"{name}_{label}"] = len(np.unique(ids))
    return fit, selection, cutoff, counts

if MODE != "check":
    print("Contrast uses fixed minimum activation; support counts distinct patients")

Contrast uses fixed minimum activation; support counts distinct patients


## Paired patient resampling

Both classes are resampled with the same draw for every method and C. Legacy then balances sampled class sizes; all-negatives retains both complete sampled pools. Two distinct patients per class are required to compute a direction.

In [3]:
def draw_multiplicity(labels, seed, method, *, joint=True):
    labels = np.asarray(labels, int)
    rng = np.random.default_rng(seed)
    pools = [np.flatnonzero(labels == label) for label in (0, 1)]
    draws = [rng.choice(pool, len(pool), replace=True) if joint else pool.copy() for pool in pools]
    if method == "legacy":
        count = min(map(len, pools))
        draws = [(np.random.default_rng(42).choice(draw,count,replace=False) if not joint and label==1 else rng.choice(draw,count,replace=False)) if len(draw)>count else draw for label,draw in enumerate(draws)]
    elif method != "all_negatives":
        raise ValueError("Unknown CAV method")
    return np.bincount(np.concatenate(draws), minlength=len(labels))

def fit_cav(X, labels, multiplicity, mass, C, patients):
    labels = np.asarray(labels, int)
    weights = class_weights(labels, multiplicity, mass)
    support = [len(np.unique(np.asarray(patients)[(labels == k) & (weights > 0)])) for k in (0, 1)]
    if min(support) < MIN_CLASS:
        return {"status": "unsupported", "support": support}
    # pragmatic: liblinear is CPU-only; independent fits use all available workers.
    model = LogisticRegression(C=C, penalty="l2", solver="liblinear", class_weight=None,
                               max_iter=1000, random_state=42)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, labels, sample_weight=weights)
    norm = np.linalg.norm(model.coef_[0])
    if any(issubclass(w.category, ConvergenceWarning) for w in caught) or not np.isfinite(norm) or norm <= 0:
        return {"status": "failed", "reason": "convergence or nonfinite/zero coefficient", "support": support}
    return {"status": "measured", "direction": model.coef_[0] / norm, "model": model,
            "support": support, "class_mass": [float(weights[labels == k].sum()) for k in (0, 1)]}

if MODE != "check":
    print("Paired multiplicities; class_weight=None; solver seed 42; convergence failures retained")

Paired multiplicities; class_weight=None; solver seed 42; convergence failures retained


## Reference discrimination and influence

Contrast AUROC and class accuracy measure held-out discrimination. Within A and R, TCAV measures sign while signed and absolute sensitivity retain magnitude. Patient intervals condition on each fitted direction; cosine spread describes fitting variation.

In [4]:
def wilson(success, total):
    if not total:
        return (np.nan, np.nan)
    z, p = 1.959963984540054, success / total
    center = (p + z*z/(2*total)) / (1 + z*z/total)
    half = z*np.sqrt(p*(1-p)/total + z*z/(4*total*total)) / (1 + z*z/total)
    return center-half, center+half

def weighted_mean(values, weights):
    values, weights = np.asarray(values, float), np.asarray(weights, float)
    valid = np.isfinite(values)
    device = ctx["device"] if ctx is not None else "cpu"
    w = torch.as_tensor(weights, dtype=torch.float64, device=device)
    v = torch.as_tensor(np.where(valid,values,0), dtype=w.dtype, device=device)
    finite = torch.as_tensor(valid,dtype=w.dtype,device=device)
    numerator, denominator = w @ v, w @ finite
    return torch.where(denominator>0,numerator/denominator,torch.nan).cpu().numpy()

def row_medians(values):
    values=np.asarray(values,float)
    result=np.full(len(values),np.nan)
    computable=np.isfinite(values).any(axis=1)
    result[computable]=np.nanmedian(values[computable],axis=1)
    return result

def interval(values):
    values = np.asarray(values, float)
    valid = values[np.isfinite(values)]
    return {"low": float(np.quantile(valid, .025)) if len(valid) else None,
            "high": float(np.quantile(valid, .975)) if len(valid) else None,
            "undefined_fraction": float(1 - len(valid)/len(values)) if len(values) else 1.}

def weighted_auc(labels,scores,weights):
    order=np.argsort(scores,kind="stable"); y=np.asarray(labels)[order]; ordered=np.asarray(scores)[order]
    device=ctx["device"] if ctx is not None else "cpu"
    w=torch.as_tensor(np.asarray(weights)[:,order],dtype=torch.float64,device=device)
    positive=torch.as_tensor(y==1,dtype=w.dtype,device=device)
    neg=w*(1-positive); pos=w*positive
    starts=np.r_[0,np.flatnonzero(np.diff(ordered))+1]; ends=np.r_[starts[1:]-1,len(order)-1]
    if not len(order):
        return np.full(len(weights),np.nan)
    cumulative=torch.cumsum(neg,dim=1)
    before=torch.cat([torch.zeros((len(w),1),dtype=w.dtype,device=device),cumulative[:,starts[1:]-1]],dim=1)
    ties=cumulative[:,ends]-before
    ranks=torch.repeat_interleave(before+.5*ties,torch.as_tensor(ends-starts+1,device=device),dim=1)
    denominator=pos.sum(dim=1)*neg.sum(dim=1)
    return torch.where(denominator>0,(pos*ranks).sum(dim=1)/denominator,torch.nan).cpu().numpy()

def discrimination(fit, unit, rows, weights):
    if fit["status"] != "measured" or not len(rows):
        return {"auroc": None, "status": "unsupported"}
    y = unit["rule_mask"][rows].astype(int)
    scores = fit["model"].predict_proba(unit["embeddings"][rows])[:, 1]
    auc = float(roc_auc_score(y, scores)) if len(np.unique(y)) == 2 else None
    draws = weighted_auc(y,scores,weights)
    result = {"status": "measured" if auc is not None else "undefined", "auroc": auc, "auroc_interval": interval(draws)}
    for label in (0, 1):
        at = y == label; n = int(at.sum()); correct = int(((scores >= .5) == y)[at].sum())
        result[f"class_{label}"] = {"n": n, "accuracy": correct/n if n else None, "wilson": wilson(correct, n)}
    return result

if MODE != "check":
    print("AUROC intervals and Wilson class intervals retain missing-class support")

AUROC intervals and Wilson class intervals retain missing-class support


In [5]:
def projected_metrics(fits, gradients, device):
    valid = [(i, fit) for i, fit in enumerate(fits) if fit["status"] == "measured"]
    if not valid:
        return np.empty((len(gradients), 0)), []
    g = torch.as_tensor(gradients, dtype=torch.float32, device=device)
    c = torch.as_tensor(np.stack([fit["direction"] for _, fit in valid]), dtype=g.dtype, device=device)
    return (g @ c.T).cpu().numpy(), [i for i, _ in valid]

def fit_spread(values):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    return {"median": float(np.median(values)) if len(values) else None,
            "q05": float(np.quantile(values, .05)) if len(values) else None,
            "q95": float(np.quantile(values, .95)) if len(values) else None}

def influence(sensitivity, mask, weights):
    values, selected_weights = sensitivity[mask], weights[:, mask]
    if not len(values) or not values.shape[1]:
        return {"status": "undefined", "n": int(mask.sum()), "fits": int(values.shape[1])}, {}
    summary, private = {"status": "measured", "n": len(values), "fits": values.shape[1]}, {}
    for name, matrix in (("tcav", (values > 0).astype(float)), ("signed", values), ("absolute", np.abs(values))):
        per_fit = matrix.mean(axis=0)
        draws = weighted_mean(matrix, selected_weights)
        headline_draws = row_medians(draws)
        summary[name] = {**fit_spread(per_fit), "patient_interval": interval(headline_draws)}
        private[name] = {"per_fit": per_fit, "draws": draws, "headline_draws": headline_draws}
        if name == "signed":
            limits = [interval(draws[:, i]) for i in range(draws.shape[1])]
            summary["positive_intervals"] = sum(v["low"] is not None and v["low"] > 0 for v in limits)
            summary["negative_intervals"] = sum(v["high"] is not None and v["high"] < 0 for v in limits)
    return summary, private

if MODE != "check":
    print("Tensor projections use selected device; median scalar metrics retain individual fit identities")

Tensor projections use selected device; median scalar metrics retain individual fit identities


## Reference experiment

The complete fixed grid contains six settings. Joint draws, three negative-only redraws and five label controls are separate records. Original historical CAVs remain a separately named baseline; every failure stays in the private ledger and aggregate coverage.

In [6]:
def reference_fits(unit, fit_rows, mass, seeds, *, joint):
    X, y = unit["embeddings"][fit_rows], unit["rule_mask"][fit_rows].astype(int)
    patients = unit["patients"][fit_rows]
    jobs = [(method, C, seed, draw_multiplicity(y, seed, method, joint=joint))
            for method in METHODS for C in C_GRID for seed in seeds]
    fitted = Parallel(n_jobs=-1)(delayed(fit_cav)(X, y, mult, mass, C, patients) for method,C,seed,mult in jobs)
    return {(method,C): [{**fit, "seed": seed} for (m,c,seed,_),fit in zip(jobs,fitted) if (m,c)==(method,C)]
            for method in METHODS for C in C_GRID}

def label_controls(unit, rows, mass):
    X, y, patients = unit["embeddings"][rows], unit["rule_mask"][rows].astype(int), unit["patients"][rows]
    return [{**fit_cav(X, np.random.default_rng(seed).permutation(y), np.ones(len(y)), mass, .1, patients),
             "seed": seed} for seed in LABEL_SEEDS]

def direction_cosines(fits):
    directions = [fit["direction"] for fit in fits if fit["status"] == "measured"]
    if len(directions) < 2:
        return {"min": None, "q10": None, "median": None}
    matrix = np.stack(directions) @ np.stack(directions).T
    values = matrix[np.triu_indices(len(matrix), 1)]
    return {"min": float(values.min()), "q10": float(np.quantile(values,.1)), "median": float(np.median(values))}

if MODE != "check":
    print(f"Independent sklearn workers: {os.cpu_count()}; all failed attempts retained")

Independent sklearn workers: 64; all failed attempts retained


In [7]:
def assess_reference(unit, device, joint_seeds):
    rows, selection, cutoff, support = contrast_support(unit)
    mass = 2 * min(support["fit_0"], support["fit_1"])
    result = {"identity": unit["identity"], "negative_cutoff": cutoff, "support": support,
              "historical": unit.get("historical"), "aliases": unit["aliases"], "reference": unit["reference"],
              "joint": {}, "negative": {}, "labels": [], "scores": {}}
    if min(support["fit_0"], support["fit_1"]) < MIN_CLASS:
        result["status"] = "unsupported"
        return result
    result["status"] = "measured"
    measured=time.monotonic()
    result["joint"] = reference_fits(unit, rows, mass, joint_seeds, joint=True)
    result["joint_seconds"]=time.monotonic()-measured
    measured=time.monotonic()
    result["negative"] = reference_fits(unit, rows, mass, NEGATIVE_SEEDS, joint=False)
    result["negative_seconds"]=time.monotonic()-measured
    measured=time.monotonic()
    result["labels"] = label_controls(unit, rows, mass)
    result["label_seconds"]=time.monotonic()-measured
    measured=time.monotonic()
    result.update(reference_assessments(unit,result,selection,device))
    result["assessment_seconds"]=time.monotonic()-measured
    return result

def reference_assessments(unit,result,selection,device):
    weights = patient_weights(unit["patients"][unit["selection"]], seeds=INTERVAL_SEEDS)
    positions = {row:i for i,row in enumerate(unit["selection"])}
    contrast_weights = weights[:,[positions[row] for row in selection]]
    for setting, fits in result["joint"].items():
        if ctx is not None:
            check_budget(ctx)
        sensitivity, fit_ids = projected_metrics(fits, unit["gradients"][unit["selection"]], device)
        strata = {name: influence(sensitivity, mask, weights) for name,mask in
                  (("A", unit["activation"][unit["selection"]] > unit["cutoff"]),
                   ("R", unit["rule_mask"][unit["selection"]]))}
        result["scores"][setting] = {"discrimination": [discrimination(f,unit,selection,contrast_weights) for f in fits],
             "fit_ids": fit_ids, "strata": strata, "cosines": direction_cosines(fits)}
    result["negative_scores"] = {setting:[discrimination(f,unit,selection,contrast_weights)["auroc"] for f in fits]
                                 for setting,fits in result["negative"].items()}
    result["label_scores"] = [discrimination(f,unit,selection,contrast_weights) for f in result["labels"]]
    return result

if MODE != "check":
    print("Reference assessment keeps support, historical fits, candidate grid, redraws and label controls")

Reference assessment keeps support, historical fits, candidate grid, redraws and label controls


## One global estimator decision

Primary last_3 families share each ranking denominator. An AUROC loss no larger than legacy's three-redraw range is admissible. Adoption needs at least one patient-unit reduction of fitting TCAV width; this is a procedure-selection convention, never a concept-quality cutoff.

In [8]:
def paired_setting(result, setting, stratum):
    baseline, candidate = result["scores"][("legacy",.1)], result["scores"][setting]
    common = sorted(set(baseline["fit_ids"]) & set(candidate["fit_ids"]))
    ba = [v.get("auroc") for v in baseline["discrimination"]]
    ca = [v.get("auroc") for v in candidate["discrimination"]]
    auc = [ca[i]-ba[i] for i in common if ca[i] is not None and ba[i] is not None]
    left, right = baseline["strata"][stratum], candidate["strata"][stratum]
    n = left[0]["n"]
    if len(common)<2 or not n or "tcav" not in left[1] or "tcav" not in right[1]:
        return None
    bmap, cmap = {v:i for i,v in enumerate(baseline["fit_ids"])}, {v:i for i,v in enumerate(candidate["fit_ids"])}
    b = left[1]["tcav"]["per_fit"][[bmap[i] for i in common]]
    c = right[1]["tcav"]["per_fit"][[cmap[i] for i in common]]
    wb, wc = np.diff(np.quantile(b,[.05,.95]))[0], np.diff(np.quantile(c,[.05,.95]))[0]
    return {"reduction": float((wb-wc)*n), "width_change": float(wc-wb),
            "auc_change": float(np.median(auc)) if auc else None, "common_fits": len(common), "n": n}

def setting_preference(setting):
    method, C = setting
    return (setting != ("legacy",.1), C != .1, C, method)

def selection_panel(results):
    primary = {key:r for key,r in results.items() if "last_3" in r["aliases"] and r["reference"]<=2013 and r["status"]=="measured"}
    settings = [(m,c) for m in METHODS for c in C_GRID]
    ranges = [max(values)-min(values) for r in primary.values()
              if len(values:=r["negative_scores"][("legacy",.1)])==3 and all(v is not None for v in values)]
    floor = float(np.median(ranges)) if ranges else None
    paired = {s:{k:{z:paired_setting(r,s,z) for z in ("A","R")} for k,r in primary.items()} for s in settings}
    common = {z:[k for k in primary if all(paired[s][k][z] is not None for s in settings)] for z in ("A","R")}
    common_auc = [k for k in primary if all((p:=paired[s][k]["A"] or paired[s][k]["R"]) is not None and p["auc_change"] is not None for s in settings)]
    return primary,settings,ranges,floor,paired,common,common_auc

def select_procedure(results):
    primary,settings,ranges,floor,paired,common,common_auc=selection_panel(results)
    ledger, strata_used = [], []
    for setting in settings:
        reductions = [float(np.median([paired[setting][k][z]["reduction"] for k in common[z]])) for z in ("A","R") if common[z]]
        strata_used = [z for z in ("A","R") if common[z]]
        changes = [(paired[setting][k]["A"] or paired[setting][k]["R"])["auc_change"] for k in common_auc]
        auc = float(np.median(changes)) if changes else None
        score = float(np.mean(reductions)) if reductions else None
        ledger.append({"method":setting[0],"C":setting[1],"reduction":score,"auc_change":auc,
                       "auc_lower_tail":float(np.quantile(changes,.1)) if changes else None,
                       "A_families":len(common["A"]),"R_families":len(common["R"]),"auc_families":len(common_auc),
                       "noise_floor":floor,"noise_families":len(ranges), "eligible":floor is not None and auc is not None and auc>=-floor and score is not None})
    admitted = [r for r in ledger if r["eligible"]]
    selected, status = ("legacy",.1), "not_computable"
    if floor is not None and strata_used and admitted:
        best = max(r["reduction"] for r in admitted)
        winner = min((r for r in admitted if r["reduction"]>=best-TIE),key=lambda r:setting_preference((r["method"],r["C"])))
        selected = (winner["method"],winner["C"]) if best>=1 else ("legacy",.1)
        status = "selected" if best>=1 else "retained_legacy"
    return {"setting":selected,"status":status,"strata":strata_used,"ledger":ledger,"paired":paired,"common":common}

if MODE != "check":
    print("Global reference decision: measured negative-redraw AUROC floor and one-patient W margin")

Global reference decision: measured negative-redraw AUROC floor and one-patient W margin


In [9]:
def clean_records(records):
    return [{key:(None if isinstance(value,(float,np.floating)) and not np.isfinite(value) else value)
             for key,value in row.items()} for row in records]

def reference_tables(unit,result):
    rows=[]
    for setting,scores in result["scores"].items():
        fits=result["joint"][setting]
        for draw,fit in enumerate(fits):
            record={"concept":unit["key"],"variants":",".join(unit["aliases"]),"reference":unit["reference"],"method":setting[0],"C":setting[1],
                    "draw":fit["seed"],"status":fit["status"],"fit_negative":fit["support"][0],"fit_positive":fit["support"][1],"reason":fit.get("reason")}
            d=scores["discrimination"][draw]
            record.update(auroc=d.get("auroc"),auroc_low=d.get("auroc_interval",{}).get("low"),
                          auroc_high=d.get("auroc_interval",{}).get("high"),auroc_undefined=d.get("auroc_interval",{}).get("undefined_fraction"))
            for label in (0,1):
                acc=d.get(f"class_{label}",{})
                record.update({f"accuracy_{label}":acc.get("accuracy"),f"accuracy_{label}_low":acc.get("wilson",(None,None))[0],
                               f"accuracy_{label}_high":acc.get("wilson",(None,None))[1],f"selection_{label}":acc.get("n",0)})
            rows.extend(reference_strata_rows(record,scores,draw))
    for setting,values in result.get("negative_scores",{}).items():
        rows.extend({"concept":unit["key"],"reference":unit["reference"],"method":setting[0],"C":setting[1],
                     "source":"negative_only","draw":seed,"auroc":value} for seed,value in zip(NEGATIVE_SEEDS,values))
    rows.extend({"concept":unit["key"],"reference":unit["reference"],"source":"label_permutation","draw":seed,
                 "auroc":value.get("auroc"),"status":value["status"]} for seed,value in zip(LABEL_SEEDS,result.get("label_scores",[])))
    for seed,fit in (result.get("historical") or {}).get("cavs",{}).items():
        rows.append({"concept":unit["key"],"reference":unit["reference"],"source":"original_unmodified_C_0.1","draw":int(seed),
                     "status":"measured" if fit.get("valid") else "unsupported","historical_cv_auroc":fit.get("cv_auroc"),
                     "historical_cv_accuracy":fit.get("cv_accuracy"),"fit_positive":fit.get("positive_training_count"),"fit_negative":fit.get("negative_count")})
    for name in ("joint_seconds","negative_seconds","label_seconds","assessment_seconds"):
        if name in result:
            rows.append({"concept":unit["key"],"reference":unit["reference"],"source":"timing","metric":name,"value":result[name]})
    return rows

if MODE != "check":
    print("Public reference ledger: every attempted fit, discrimination and subgroup uncertainty")

Public reference ledger: every attempted fit, discrimination and subgroup uncertainty


In [10]:
def reference_strata_rows(record,scores,draw):
    rows=[]
    for name,(summary,private) in scores["strata"].items():
        row={**record,"source":"joint","subgroup":name,"n":summary["n"],"valid_fits":summary["fits"],
             **{f"cosine_{k}":v for k,v in scores["cosines"].items()}}
        tcav=summary.get("tcav",{})
        width=tcav.get("q95")-tcav.get("q05") if tcav.get("q95") is not None else None
        row.update(W=width,W_times_n=width*summary["n"] if width is not None else None)
        if draw in scores["fit_ids"] and private:
            column=scores["fit_ids"].index(draw)
            for metric in ("tcav","signed","absolute"):
                point=private[metric]["per_fit"][column]; ci=interval(private[metric]["draws"][:,column])
                row.update({metric:float(point),metric+"_low":ci["low"],metric+"_high":ci["high"],metric+"_undefined":ci["undefined_fraction"]})
        rows.append(row)
    return rows

if MODE != "check":
    print("W and W*n accompany per-fit TCAV, signed/absolute sensitivity and signed cosine spread")

W and W*n accompany per-fit TCAV, signed/absolute sensitivity and signed cosine spread


In [11]:
def endpoint_table(decision,results):
    rows=[]
    for setting,families in decision["paired"].items():
        for key,strata in families.items():
            for subgroup,value in strata.items():
                if value is not None:
                    rows.append({"source":"family_endpoint","concept":key,"reference":results[key]["reference"],"method":setting[0],"C":setting[1],
                                 "subgroup":subgroup,"W_candidate_minus_legacy":value["width_change"],
                                 "W_reduction_times_n":value["reduction"],"common_fits":value["common_fits"],
                                 "ranked_family":key in decision["common"][subgroup]})
    if rows:
        frame=pd.DataFrame(rows); ranked=frame[frame["ranked_family"]]
        for keys,part in ranked.groupby(["method","C","subgroup","reference"]):
            method,C,subgroup,reference=keys
            total=sum("last_3" in r["aliases"] and r["reference"]==reference for r in results.values())
            rows.append({"source":"reference_endpoint","method":method,"C":C,"subgroup":subgroup,"reference":int(reference),
                         "W_candidate_minus_legacy":float(part["W_candidate_minus_legacy"].median()),"n_families":len(part),"total_families":total})
        for keys,part in ranked.groupby(["method","C","subgroup"]):
            method,C,subgroup=keys
            total=sum("last_3" in r["aliases"] and r["reference"]<=2013 for r in results.values())
            rows.append({"source":"overall_endpoint","method":method,"C":C,"subgroup":subgroup,
                         "W_candidate_minus_legacy":float(part["W_candidate_minus_legacy"].median()),"n_families":len(part),"total_families":total})
    return rows

if MODE != "check":
    print("Primary endpoint: paired W(candidate)-W(legacy), separately A/R, with ranking denominators")

Primary endpoint: paired W(candidate)-W(legacy), separately A/R, with ranking denominators


## Develop and freeze

Development never reads evaluation rows or later observations. Every family's original eligibility remains visible; unsupported families remain in coverage. The branch freeze contains all attempted private fits and the globally selected setting.

In [12]:
reference_results, support_records, candidate_records = {}, [], []
fit_records, selection_records, diagnostic_records = [], [], []
if MODE in {"develop", "smoke"}:
    for unit in iter_units(ctx, "measurement"):
        check_budget(ctx)
        measured_start=time.monotonic()
        result = assess_reference(unit, ctx["device"], JOINT_SEEDS[:2] if MODE=="smoke" else JOINT_SEEDS)
        result["seconds"]=time.monotonic()-measured_start
        reference_results[unit["key"]] = result
        with (ctx["out"]/"private"/f"reference-{len(reference_results):04d}.pkl").open("xb") as handle:
            pickle.dump(result,handle)
        support_records.append({"concept":unit["key"],"variants":",".join(unit["aliases"]),"reference":unit["reference"],"family":unit["family"],
           "status":result["status"],"fit_positive":result["support"]["fit_1"],"fit_negative":result["support"]["fit_0"],
           "selection_positive":result["support"]["selection_1"],"selection_negative":result["support"]["selection_0"],
           "negative_cutoff":result["negative_cutoff"] if np.isfinite(result["negative_cutoff"]) else None,
           "original_idea1":bool(unit["eligible"].get("idea1",False)),"original_idea2":bool(unit["eligible"].get("idea2",False)),
           "seconds":result["seconds"]})
        fit_records.extend(reference_tables(unit,result))
        display(pd.DataFrame(support_records[-1:]))
    decision = select_procedure(reference_results)
    selection_records=decision["ledger"]
    candidate_records=endpoint_table(decision,reference_results)
    display(pd.DataFrame(selection_records))
    display(pd.DataFrame(fit_records))
    if MODE == "develop":
        freeze_write(ctx, {"units":reference_results,"inputs":{k:v["identity"] for k,v in reference_results.items()},"selected":decision})
    print(f"{len(reference_results)} unique family/builds; decision {decision['setting']}; {decision['status']}")

## Annual A/R and frozen ensemble trajectories

Each headline is the median of per-fit scalar metrics. Separate 5th–95th percentiles show fitting spread. Shared patient draws retain temporal dependence and overlapping concepts; 2015 contributes no mortality outcomes.

In [13]:
def binary_metrics(target, selected, weights=None):
    target, selected = np.asarray(target,bool), np.asarray(selected,bool)
    w = np.ones(len(target)) if weights is None else np.asarray(weights,float)
    tp = float(w[target & selected].sum()); fp=float(w[~target & selected].sum()); fn=float(w[target & ~selected].sum())
    total, positives, selected_n = float(w.sum()), float(w[target].sum()), float(w[selected].sum())
    return {"tp":tp,"fp":fp,"fn":fn,"precision":tp/(tp+fp) if tp+fp else np.nan,
            "recall":tp/(tp+fn) if tp+fn else np.nan,"f2":5*tp/(5*tp+4*fn+fp) if 5*tp+4*fn+fp else np.nan,
            "A_prevalence":positives/total if total else np.nan,"R_prevalence":selected_n/total if total else np.nan,
            "P_A_given_R":tp/selected_n if selected_n else np.nan,"overlap":tp/(tp+fp+fn) if tp+fp+fn else np.nan}

def annual_summary(unit, fits, year, weights, sensitivity):
    at = (unit["years"]==year) & (unit["roles"]=="evaluation")
    A, R = unit["activation"]>unit["cutoff"], unit["rule_mask"]
    result = {"year":year,"n":int(at.sum()),"A_n":int((at&A).sum()),"R_n":int((at&R).sum()),
              "binary":binary_metrics(A[at],R[at]),"activation":float(unit["activation"][at].mean()) if at.any() else None}
    result["binary_intervals"] = {name:interval([binary_metrics(A[at],R[at],w[at])[name] for w in weights])
                                   for name in result["binary"]}
    result["activation_interval"] = interval(weighted_mean(unit["activation"][at,None],weights[:,at])[:,0])
    result["subgroups"], private = {}, {}
    norm = np.linalg.norm(unit["gradients"],axis=1)
    for name,mask in (("A",at&A),("R",at&R)):
        summary, values = influence(sensitivity,mask,weights)
        summary["zero_gradients"] = int((norm[mask]==0).sum())
        normalized = np.divide(sensitivity[mask],norm[mask,None],out=np.full_like(sensitivity[mask],np.nan),where=norm[mask,None]>0)
        summary["normalized_projection"] = fit_spread(np.divide(np.nansum(normalized,axis=0),np.isfinite(normalized).sum(axis=0),out=np.full(normalized.shape[1],np.nan),where=np.isfinite(normalized).sum(axis=0)>0)) if len(normalized) and normalized.shape[1] else fit_spread([])
        if year<=OUTCOME_LAST_YEAR and mask.any():
            y, p = unit["labels"][mask], np.asarray(unit["probabilities"])[mask]
            p = p[:,1] if p.ndim==2 else p
            summary["deaths"] = int(y.sum()); summary["brier"] = float(np.mean((p-y)**2))
            summary["mortality"] = float(y.mean())
            summary["brier_interval"] = interval(weighted_mean(((p-y)**2)[:,None],weights[:,mask])[:,0])
            summary["mortality_interval"] = interval(weighted_mean(y[:,None],weights[:,mask])[:,0])
        result["subgroups"][name], private[name] = summary, values
    return result,private

if MODE != "check":
    print("Annual conditional intervals use shared patient multiplicities; 2015 labels excluded")

Annual conditional intervals use shared patient multiplicities; 2015 labels excluded


## Returners, observation entrants and membership transitions

Patients are paired before A/R strata. Reference-to-year and consecutive-calendar-year comparisons remain distinct; missing years stay gaps. Population means reconcile with the returner/entrant/nonreturner mixture.

In [14]:
def paired_rows(unit, start, end):
    role = unit["roles"] == "evaluation"
    left=np.flatnonzero(role & (unit["years"]==start)); right=np.flatnonzero(role & (unit["years"]==end))
    lp={str(unit["patients"][i]):i for i in left}; rp={str(unit["patients"][i]):i for i in right}
    if len(lp)!=len(left) or len(rp)!=len(right):
        raise ValueError("Duplicate patient within year")
    ids=sorted(lp.keys() & rp.keys())
    return (left,right,np.array([lp[p] for p in ids],int),np.array([rp[p] for p in ids],int),
            np.array([rp[p] for p in sorted(rp.keys()-lp.keys())],int),np.array([lp[p] for p in sorted(lp.keys()-rp.keys())],int))

def paired_effect(sensitivity,left,right,weights):
    result={}
    if not len(left) or not sensitivity.shape[1]:
        return result
    for metric,values in (("tcav",(sensitivity>0).astype(float)),("signed",sensitivity),("absolute",np.abs(sensitivity))):
        before,after=values[left],values[right]
        difference=(after-before).mean(axis=0)
        draws=weighted_mean(after-before,weights[:,left])
        result[metric]={"paired_median":float(np.median(difference)),"n":len(left),
            "difference_of_medians":float(np.median(after.mean(axis=0))-np.median(before.mean(axis=0))),
            "patient_interval":interval(row_medians(draws)),**fit_spread(difference)}
    return result

def composition(unit,start,end,weights,sensitivity):
    left,right,a,b,entrants,nonreturners=paired_rows(unit,start,end)
    activation=np.asarray(unit["activation"])
    mean=lambda rows:float(activation[rows].mean()) if len(rows) else None
    result={"start":start,"end":end,"returners":len(a),"entrants":len(entrants),"nonreturners":len(nonreturners),
            "start_n":len(left),"end_n":len(right),"start_mean":mean(left),"end_mean":mean(right),
            "returner_start_mean":mean(a),"returner_end_mean":mean(b),"entrant_mean":mean(entrants),"nonreturner_mean":mean(nonreturners),
            "first_observed_entrants":int((unit["first_observed_year"][entrants]==end).sum()),
            "previously_observed_entrants":int((unit["first_observed_year"][entrants]<end).sum())}
    result["paired_activation_change"]=float((activation[b]-activation[a]).mean()) if len(a) else None
    result["paired_activation_interval"]=interval(weighted_mean((activation[b]-activation[a])[:,None],weights[:,a])[:,0])
    result["transitions"], result["paired_influence"]={},{}
    for name,mask in (("A",activation>unit["cutoff"]),("R",unit["rule_mask"])):
        result["transitions"][name]={f"{i}{j}":int(((mask[a]==i)&(mask[b]==j)).sum()) for i in (0,1) for j in (0,1)}
    result["binary_changes"]={}
    for metric in ("A_prevalence","R_prevalence","P_A_given_R","f2"):
        A=activation>unit["cutoff"]; R=unit["rule_mask"]
        point=binary_metrics(A[b],R[b])[metric]-binary_metrics(A[a],R[a])[metric]
        draws=[binary_metrics(A[b],R[b],w[a])[metric]-binary_metrics(A[a],R[a],w[a])[metric] for w in weights]
        result["binary_changes"][metric]={"change":float(point),"patient_interval":interval(draws)}
    result["paired_influence"]=paired_effect(sensitivity,a,b,weights)
    result["transition_influence"]={}
    for name,mask in (("A",activation>unit["cutoff"]),("R",unit["rule_mask"])):
        for i in (0,1):
            for j in (0,1):
                chosen=(mask[a]==i)&(mask[b]==j)
                result["transition_influence"][f"{name}_{i}{j}"]=paired_effect(sensitivity,a[chosen],b[chosen],weights)
    return result

if MODE != "check":
    print("Comparison endpoints determine observation categories; pairing precedes membership transitions")

Comparison endpoints determine observation categories; pairing precedes membership transitions


## Residual-preserving factor removal and numerical replay

Remove the original medoid contribution directly from raw embeddings, preserving residual, bias and other factors. A random direction with the same per-row norm is a separate control. Replay must agree with cached probabilities and gradients before interpreting effects.

In [15]:
def removal_embeddings(embeddings, activation, decoder, scale, seed=42):
    h=np.asarray(embeddings); component=np.asarray(activation)[:,None]*(np.asarray(decoder)*np.asarray(scale))
    direction=np.random.default_rng(seed).normal(size=h.shape)
    direction/=np.maximum(np.linalg.norm(direction,axis=1,keepdims=True),1e-12)
    control=direction*np.linalg.norm(component,axis=1,keepdims=True)
    return h-component,h+control,component

def double_probability(continuation, embeddings):
    h=embeddings
    logits=continuation.head(h[:,None,:])[...,:2]
    logits=logits / torch.exp(torch.tensor(continuation.temperature,dtype=h.dtype,device=h.device))
    return continuation.aggregate(continuation.parse(logits,continuation.configuration),continuation.reweight,continuation.eval_ys)

def numerical_replay(unit,rows):
    from concept_subgroup_drift_pilot import batched_death_probabilities, batched_death_gradients
    continuation=unit["continuation"]; h=unit["embeddings"][rows]
    if tuple(continuation.classes)!=(0,1) or not np.isfinite(continuation.temperature):
        raise RuntimeError("Mortality class order/temperature mismatch")
    p=batched_death_probabilities(continuation,h,16); g=batched_death_gradients(continuation,h,4)
    np.testing.assert_allclose(p,unit["probabilities"][rows],rtol=1e-4,atol=1e-5)
    np.testing.assert_allclose(g,unit["gradients"][rows],rtol=1e-3,atol=1e-5)
    audit=deepcopy(continuation); audit.head.double()
    place=next(audit.head.parameters()).device
    x=torch.as_tensor(h[:8],dtype=torch.float64,device=place).requires_grad_(True)
    d=torch.as_tensor(np.random.default_rng(42).normal(size=x.shape),dtype=x.dtype,device=place)
    d=d/torch.linalg.vector_norm(d,dim=1,keepdim=True)
    exact,=torch.autograd.grad(double_probability(audit,x)[:,1].sum(),x)
    with torch.no_grad():
        fd=(double_probability(audit,x+1e-3*d)[:,1]-double_probability(audit,x-1e-3*d)[:,1])/(2e-3)
    projected=(exact*d).sum(dim=1)
    if not torch.all(torch.abs(projected-fd)<=1e-8+1e-4*torch.abs(fd)):
        raise RuntimeError("Independent double finite difference mismatch")
    cpu=deepcopy(continuation); cpu.head.cpu()
    if torch.is_tensor(cpu.eval_ys):
        cpu.eval_ys=cpu.eval_ys.cpu()
    cpu.configuration=[v.cpu() if torch.is_tensor(v) else v for v in cpu.configuration]
    cpu_p=batched_death_probabilities(cpu,h,16)
    np.testing.assert_allclose(p,cpu_p,rtol=1e-4,atol=1e-5)
    direction=d.detach().cpu().numpy()
    mismatches=np.sum(np.sign((g[:8]*direction).sum(1))!=np.sign((unit["gradients"][rows][:8]*direction).sum(1)))
    return {"probability_error":float(np.max(np.abs(p-unit["probabilities"][rows]))),
            "gradient_error":float(np.max(np.abs(g-unit["gradients"][rows]))),"projected_sign_disagreement":int(mismatches),
            "cpu_gpu_probability_error":float(np.max(np.abs(p-cpu_p))),
            "finite_difference_error":float(torch.max(torch.abs(projected-fd)).detach().cpu())}

if MODE != "check":
    print("Replay: cached CPU parity plus independent double finite differences; no silent device fallback")

Replay: cached CPU parity plus independent double finite differences; no silent device fallback


In [16]:
def removal_effects(unit,year,mask,weights):
    from concept_subgroup_drift_pilot import batched_death_probabilities
    at=np.flatnonzero((unit["years"]==year)&(unit["roles"]=="evaluation")&mask)
    if not len(at):
        return {"status":"undefined","n":0}
    removed,control,component=removal_embeddings(unit["embeddings"][at],unit["activation"][at],
                              unit["decoders"][unit["seed"]][unit["factor"]],unit["scale"])
    continuation=unit["continuation"]
    original=batched_death_probabilities(continuation,unit["embeddings"][at],16)
    baseline=unit["probabilities"][at]
    np.testing.assert_allclose(original,baseline,rtol=1e-4,atol=1e-5)
    zero,_,_=removal_embeddings(unit["embeddings"][at],np.zeros(len(at)),unit["decoders"][unit["seed"]][unit["factor"]],unit["scale"])
    np.testing.assert_array_equal(zero,unit["embeddings"][at])
    result={"status":"measured","n":len(at)}
    for name,changed in (("removed",removed),("norm_control",control)):
        p=batched_death_probabilities(continuation,changed,16)[:,1]
        difference=p-baseline[:,1]
        record={"probability_change":float(difference.mean()),"probability_interval":interval(weighted_mean(difference[:,None],weights[:,at])[:,0])}
        if year<=OUTCOME_LAST_YEAR:
            y=unit["labels"][at]; loss=(p-y)**2-(baseline[:,1]-y)**2
            record.update(brier_change=float(loss.mean()),brier_interval=interval(weighted_mean(loss[:,None],weights[:,at])[:,0]))
        result[name]=record
    return result

if MODE != "check":
    print("Original-factor removal and norm-matched control scored separately, with conditional intervals")

Original-factor removal and norm-matched control scored separately, with conditional intervals


In [17]:
def yearly_evidence(unit,year):
    recipe=unit.get("yearly_recipe")
    if not recipe or "yearly_description" not in unit:
        return {"status":"missing_dependency","reason":"frozen yearly rule policy unavailable"}
    fitting=np.flatnonzero((unit["years"]==year)&np.isin(unit["roles"],["rule_discovery","fitting"]))
    evaluation=np.flatnonzero((unit["years"]==year)&(unit["roles"]=="evaluation"))
    return unit["yearly_description"](unit["X"],unit["activation"],unit["cutoff"],unit["patients"],fitting,evaluation,
             unit["feature_names"],recipe,year,unit["family"])

def trajectory_rows(unit,year,subgroup,record):
    rows=[]
    for metric,details in record["per_fit"][subgroup].items():
        for i,value in enumerate(details["per_fit"]):
            ci=interval(details["draws"][:,i])
            rows.append({"concept":unit["key"],"reference":unit["reference"],"year":year,"subgroup":subgroup,
                "source":"frozen_fit_trajectory","draw":record["fit_seeds"][i],"metric":metric,"value":float(value),
                "low":ci["low"],"high":ci["high"],"undefined_fraction":ci["undefined_fraction"]})
    return rows

def evaluation_tables(unit,annual,comparisons):
    rows=[]
    for year,record in annual.items():
        summary=record["summary"]
        for metric,value in summary["binary"].items():
            ci=summary["binary_intervals"][metric]
            rows.append({"concept":unit["key"],"reference":unit["reference"],"year":year,"source":"annual_binary", "metric":metric,
                         "value":value,"low":ci["low"],"high":ci["high"],"undefined_fraction":ci["undefined_fraction"],"n":summary["n"]})
        for subgroup,part in summary["subgroups"].items():
            base={"concept":unit["key"],"reference":unit["reference"],"year":year,"subgroup":subgroup,"n":part["n"],"source":"annual"}
            deaths=part.get("deaths")
            rows.append({**base,"source":"annual_subgroup_support","deaths":deaths,
                "survivors":part["n"]-deaths if deaths is not None else None,"mortality":part.get("mortality"),
                "mortality_low":part.get("mortality_interval",{}).get("low"),"mortality_high":part.get("mortality_interval",{}).get("high"),
                "brier":part.get("brier"),"brier_low":part.get("brier_interval",{}).get("low"),"brier_high":part.get("brier_interval",{}).get("high"),
                "outcome_status":"measured" if deaths is not None else "undefined",
                "outcome_reason":"year_after_2014" if year>OUTCOME_LAST_YEAR else "empty_subgroup" if not part["n"] else None,
                "zero_norm_rows":part["zero_gradients"],"norm_quantity":"raw_probability_gradient","valid_fits":part["fits"],
                "positive_intervals":part.get("positive_intervals"),"negative_intervals":part.get("negative_intervals")})
            rows.extend(trajectory_rows(unit,year,subgroup,record))
            for metric in ("tcav","signed","absolute","normalized_projection"):
                value=part.get(metric,{})
                rows.append({**base,"metric":metric,"value":value.get("median"),"q05":value.get("q05"),"q95":value.get("q95"),
                             "low":value.get("patient_interval",{}).get("low"),"high":value.get("patient_interval",{}).get("high"),
                             "undefined_fraction":value.get("patient_interval",{}).get("undefined_fraction"),"valid_fits":part["fits"]})
            for metric,value in part.get("removal",{}).items():
                if isinstance(value,dict):
                    rows.append({**base,"source":"removal","metric":metric,"value":value["probability_change"],
                                 "low":value["probability_interval"]["low"],"high":value["probability_interval"]["high"],
                                 "brier_change":value.get("brier_change"),"brier_low":value.get("brier_interval",{}).get("low"),"brier_high":value.get("brier_interval",{}).get("high")})
        yearly=summary["yearly_description"]
        rows.append({"concept":unit["key"],"reference":unit["reference"],"year":year,"source":"yearly_rule","status":yearly["status"],
                     "value":yearly.get("independent_fidelity",{}).get("f2")})
    rows.extend(comparison_tables(unit,comparisons))
    return rows

if MODE != "check":
    print("Frozen-policy yearly descriptions are complementary; original R unchanged")

Frozen-policy yearly descriptions are complementary; original R unchanged


In [18]:
def comparison_tables(unit,comparisons):
    rows=[]
    for comparison in comparisons:
        base={"concept":unit["key"],"reference":unit["reference"],"start":comparison["start"],"year":comparison["end"],"n":comparison["returners"]}
        for metric,value in comparison["binary_changes"].items():
            rows.append({**base,"source":"paired_binary","metric":metric,"value":value["change"],
                         "low":value["patient_interval"]["low"],"high":value["patient_interval"]["high"],
                         "undefined_fraction":value["patient_interval"]["undefined_fraction"]})
        for metric,value in comparison["paired_influence"].items():
            rows.append({**base,"source":"paired_influence","metric":metric,"value":value["paired_median"],
                         "difference_of_medians":value["difference_of_medians"],"q05":value["q05"],"q95":value["q95"],
                         "low":value["patient_interval"]["low"],"high":value["patient_interval"]["high"]})
        for subgroup,metrics in comparison["transition_influence"].items():
            for metric,value in metrics.items():
                rows.append({**base,"source":"paired_transition_influence","subgroup":subgroup,"metric":metric,"value":value["paired_median"],
                             "n":value["n"],"low":value["patient_interval"]["low"],"high":value["patient_interval"]["high"]})
        for subgroup,transitions in comparison["transitions"].items():
            rows.extend({**base,"source":"membership_transition","subgroup":subgroup,"metric":state,"value":count} for state,count in transitions.items())
        for metric in ("start_mean","end_mean","returner_start_mean","returner_end_mean","entrant_mean","nonreturner_mean"):
            rows.append({**base,"source":"composition","metric":metric,"value":comparison[metric]})
    return rows

if MODE != "check":
    print("Public paired differences, membership transitions and population-mixture accounting")

Public paired differences, membership transitions and population-mixture accounting


In [19]:
if MODE == "smoke" and reference_results:
    measured=time.monotonic()
    probe=np.array(unit["cav_fit"][:8],int)
    replay=numerical_replay(unit,probe) if len(probe) else {"status":"undefined"}
    display(pd.DataFrame([{"source":"reference_replay","seconds":time.monotonic()-measured,**replay}]))
    print("Replay timing uses fitting rows only; future pairing uses synthetic Checks, not real future observations")

## Evaluation and complementary descriptions

A verified branch freeze supplies all alternative CAV fits. A/R trajectories remain available even without valid influence directions. Existing yearly rule descriptions use their previously frozen policy and protected discovery roles; they never replace R.

In [20]:
annual_records, pair_records, private_evaluation = [], [], {}
if MODE == "evaluate":
    frozen=freeze_read(ctx); selected=tuple(frozen["selected"]["setting"])
    for unit in iter_units(ctx,"measurement"):
        if unit["key"] not in frozen["units"] or unit["identity"] != frozen["inputs"][unit["key"]]:
            raise ValueError("Evaluation input identity differs from branch freeze")
        check_budget(ctx)
        bundle=frozen["units"][unit["key"]]
        fits=bundle["joint"].get(selected,[])
        sensitivity,fit_ids=projected_metrics(fits,unit["gradients"],ctx["device"])
        evaluated=unit["evaluation"]
        weights=np.zeros((len(INTERVAL_SEEDS),len(unit["patients"])),dtype=int)
        weights[:,evaluated]=patient_weights(unit["patients"][evaluated],seeds=INTERVAL_SEEDS)
        replay=numerical_replay(unit,evaluated[:16]) if len(evaluated) else {"status":"undefined"}
        annual,comparisons={},[]
        for year in range(unit["reference"],2016):
            check_budget(ctx)
            summary,details=annual_summary(unit,fits,year,weights,sensitivity)
            summary["yearly_description"]=yearly_evidence(unit,year)
            annual[year]={"summary":summary,"per_fit":details,"fit_seeds":[fits[i]["seed"] for i in fit_ids]}
            for name,mask in (("A",unit["activation"]>unit["cutoff"]),("R",unit["rule_mask"])):
                summary["subgroups"][name]["removal"]=removal_effects(unit,year,mask,weights)
            annual_records.append({"concept":unit["key"],"variants":",".join(unit["aliases"]),"reference":unit["reference"],"family":unit["family"],
               "year":year,"lag":year-unit["reference"],"n":summary["n"],"A_n":summary["A_n"],"R_n":summary["R_n"],
               "activation":summary["activation"],"f2":summary["binary"]["f2"],"P_A_given_R":summary["binary"]["P_A_given_R"],
               "A_tcav":summary["subgroups"]["A"].get("tcav",{}).get("median"),"R_tcav":summary["subgroups"]["R"].get("tcav",{}).get("median"),
               "valid_fits":len(fit_ids),"attempted_fits":len(fits)})
        endpoints=sorted({(unit["reference"],y) for y in range(unit["reference"]+1,2016)} | {(y-1,y) for y in range(unit["reference"]+1,2016)})
        for start,end in endpoints:
            comparison=composition(unit,start,end,weights,sensitivity); comparisons.append(comparison)
            pair_records.append({"concept":unit["key"],"start":start,"end":end,"lag":end-unit["reference"],
                **{k:comparison[k] for k in ("returners","entrants","nonreturners","first_observed_entrants","previously_observed_entrants","paired_activation_change")}})
        private_evaluation[unit["key"]]={"annual":annual,"comparisons":comparisons,"fit_ids":fit_ids,"replay":replay}
        with (ctx["out"]/"private"/f"evaluation-{len(private_evaluation):04d}.pkl").open("xb") as handle:
            pickle.dump(private_evaluation[unit["key"]],handle)
        diagnostic_records.extend(evaluation_tables(unit,annual,comparisons))
        display(pd.DataFrame([r for r in annual_records if r["concept"]==unit["key"]]))
    import pickle
    with (ctx["out"] / "private" / "evaluation.pkl").open("wb") as handle:
        pickle.dump(private_evaluation,handle)
    display(pd.DataFrame(pair_records))

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2007,0,237,21,30,0.456357,0.833333,0.633333,1.0,1.0,50,50
1,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2008,1,282,0,34,0.000000,0.000000,0.000000,NaN,1.0,50,50
2,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2009,2,296,0,40,0.000000,0.000000,0.000000,NaN,1.0,50,50
3,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2010,3,323,0,40,0.000000,0.000000,0.000000,NaN,1.0,50,50
4,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2011,4,319,0,46,0.000000,0.000000,0.000000,NaN,1.0,50,50
5,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2012,5,311,0,50,0.000000,0.000000,0.000000,NaN,1.0,50,50
6,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2013,6,332,0,55,0.000000,0.000000,0.000000,NaN,1.0,50,50
7,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2014,7,326,0,46,0.000000,0.000000,0.000000,NaN,1.0,50,50
8,"r1/2007/2007/42/42:94,135:3,215:146","last_3,all_history,baseline",2007,"2007/42/42:94,135:3,215:146",2015,8,287,0,38,0.000000,0.000000,0.000000,NaN,1.0,50,50


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2007,0,237,65,30,0.471791,0.327586,0.633333,0.0,0.0,50,50
1,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2008,1,282,281,37,3.096994,0.159345,1.000000,0.0,0.0,50,50
2,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2009,2,296,296,45,3.434763,0.183076,1.000000,0.0,0.0,50,50
3,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2010,3,323,323,37,3.440430,0.139202,1.000000,0.0,0.0,50,50
4,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2011,4,319,319,36,3.591929,0.137195,1.000000,0.0,0.0,50,50
5,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2012,5,311,311,37,3.758446,0.144418,1.000000,0.0,0.0,50,50
6,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2013,6,332,332,46,3.881373,0.167394,1.000000,0.0,0.0,50,50
7,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2014,7,326,326,47,3.927637,0.173945,1.000000,0.0,0.0,50,50
8,"r1/2007/2007/42/42:65,135:187,215:78","last_3,all_history,baseline",2007,"2007/42/42:65,135:187,215:78",2015,8,287,287,15,4.005820,0.064488,1.000000,0.0,0.0,50,50


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2007,0,237,25,7,0.432266,0.233645,0.714286,0.0,0.0,49,50
1,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2008,1,282,91,10,1.117206,0.133690,1.000000,0.0,0.0,49,50
2,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2009,2,296,98,8,1.211986,0.100000,1.000000,0.0,0.0,49,50
3,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2010,3,323,120,11,1.211620,0.112016,1.000000,0.0,0.0,49,50
4,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2011,4,319,164,8,1.383156,0.060241,1.000000,0.0,0.0,49,50
5,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2012,5,311,208,6,1.612492,0.035800,1.000000,0.0,0.0,49,50
6,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2013,6,332,249,12,1.791579,0.059524,1.000000,0.0,0.0,49,50
7,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2014,7,326,241,8,1.810130,0.041152,1.000000,0.0,0.0,49,50
8,"r1/2007/2007/42/42:268,135:171,215:262","last_3,all_history,baseline",2007,"2007/42/42:268,135:171,215:262",2015,8,287,208,2,1.832155,0.011990,1.000000,0.0,0.0,49,50


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2007,0,237,27,9,0.449,0.213675,0.555556,0.0,0.0,49,50
1,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2008,1,282,0,10,0.000,0.000000,0.000000,NaN,0.0,49,50
2,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2009,2,296,0,10,0.000,0.000000,0.000000,NaN,0.0,49,50
3,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2010,3,323,0,12,0.000,0.000000,0.000000,NaN,0.0,49,50
4,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2011,4,319,0,8,0.000,0.000000,0.000000,NaN,0.0,49,50
5,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2012,5,311,0,13,0.000,0.000000,0.000000,NaN,0.0,49,50
6,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2013,6,332,0,15,0.000,0.000000,0.000000,NaN,0.0,49,50
7,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2014,7,326,0,10,0.000,0.000000,0.000000,NaN,0.0,49,50
8,"r1/2007/2007/42/42:1,135:175,215:120","last_3,all_history,baseline",2007,"2007/42/42:1,135:175,215:120",2015,8,287,0,3,0.000,0.000000,0.000000,NaN,0.0,49,50


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2008,0,298,14,23,0.611565,0.886076,0.608696,0.0,0.0,50,50
1,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2009,1,314,0,20,0.313082,0.000000,0.000000,NaN,0.0,50,50
2,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2010,2,333,0,25,0.253195,0.000000,0.000000,NaN,0.0,50,50
3,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2011,3,325,0,24,0.178768,0.000000,0.000000,NaN,0.0,50,50
4,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2012,4,310,0,25,0.113435,0.000000,0.000000,NaN,0.0,50,50
5,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2013,5,314,0,22,0.056080,0.000000,0.000000,NaN,0.0,50,50
6,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2014,6,309,0,26,0.042338,0.000000,0.000000,NaN,0.0,50,50
7,"all_history/2008/2008/42/42:283,135:131,215:29","last_3,all_history",2008,"2008/42/42:283,135:131,215:29",2015,7,284,0,7,0.000040,0.000000,0.000000,NaN,0.0,50,50


... (135 outputs left out)

,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2013/2013/42/42:173,135:24,215:49",baseline,2013,"2013/42/42:173,135:24,215:49",2013,0,322,29,18,0.638152,0.335821,0.5,0.0,0.0,50,50
1,"r1/2013/2013/42/42:173,135:24,215:49",baseline,2013,"2013/42/42:173,135:24,215:49",2014,1,324,0,22,0.000000,0.000000,0.0,NaN,0.0,50,50
2,"r1/2013/2013/42/42:173,135:24,215:49",baseline,2013,"2013/42/42:173,135:24,215:49",2015,2,293,0,20,0.000000,0.000000,0.0,NaN,0.0,50,50


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2013/2013/42/42:212,135:276,215:238",baseline,2013,"2013/42/42:212,135:276,215:238",2013,0,322,32,19,0.488683,0.272109,0.421053,0.0,0.0,50,50
1,"r1/2013/2013/42/42:212,135:276,215:238",baseline,2013,"2013/42/42:212,135:276,215:238",2014,1,324,0,20,0.000000,0.000000,0.000000,NaN,0.0,50,50
2,"r1/2013/2013/42/42:212,135:276,215:238",baseline,2013,"2013/42/42:212,135:276,215:238",2015,2,293,0,17,0.000000,0.000000,0.000000,NaN,0.0,50,50


,concept,variants,reference,family,year,lag,n,A_n,R_n,activation,f2,P_A_given_R,A_tcav,R_tcav,valid_fits,attempted_fits
0,"r1/2013/2013/42/42:273,135:220,215:35",baseline,2013,"2013/42/42:273,135:220,215:35",2013,0,322,47,18,0.287014,0.339806,0.777778,1.0,1.0,50,50
1,"r1/2013/2013/42/42:273,135:220,215:35",baseline,2013,"2013/42/42:273,135:220,215:35",2014,1,324,197,21,0.815943,0.129790,1.000000,1.0,1.0,50,50
2,"r1/2013/2013/42/42:273,135:220,215:35",baseline,2013,"2013/42/42:273,135:220,215:35",2015,2,293,248,8,1.014500,0.040000,1.000000,1.0,1.0,50,50


,concept,start,end,lag,returners,entrants,nonreturners,first_observed_entrants,previously_observed_entrants,paired_activation_change
0,"r1/2007/2007/42/42:94,135:3,215:146",2007,2008,1,192,90,45,90,0,-0.467970
1,"r1/2007/2007/42/42:94,135:3,215:146",2007,2009,2,165,131,72,67,64,-0.457135
2,"r1/2007/2007/42/42:94,135:3,215:146",2007,2010,3,156,167,81,67,100,-0.453770
3,"r1/2007/2007/42/42:94,135:3,215:146",2007,2011,4,98,221,139,83,138,-0.488848
4,"r1/2007/2007/42/42:94,135:3,215:146",2007,2012,5,88,223,149,48,175,-0.509613
...,...,...,...,...,...,...,...,...,...,...
1032,"r1/2013/2013/42/42:212,135:276,215:238",2013,2015,2,227,66,95,12,54,-0.494343
1033,"r1/2013/2013/42/42:212,135:276,215:238",2014,2015,2,265,28,59,12,16,0.000000
1034,"r1/2013/2013/42/42:273,135:220,215:35",2013,2014,1,260,64,62,42,22,0.561089
1035,"r1/2013/2013/42/42:273,135:220,215:35",2013,2015,2,227,66,95,12,54,0.775136


## Findings

Tables retain denominators, unsupported cases and the original universe. Fitting spread and conditional patient intervals concern different uncertainties. Paired patients remain selected by survival and record availability; all temporal findings are exploratory.

In [21]:
if MODE != "check":
    records=annual_records if MODE=="evaluate" else support_records
    if records:
        fields=set(records[0])
        frame=publish(ctx,clean_records(records),fields)
        display(frame)
    for table in (fit_records,selection_records,candidate_records,diagnostic_records,pair_records):
        if table:
            fields=set().union(*(r.keys() for r in table))
            display(publish(ctx,clean_records(table),fields))
    if MODE=="smoke":
        print("Smoke only: two joint draws; global decision is timing-only, never a scientific freeze")
    if MODE=="evaluate" and annual_records:
        frame=pd.DataFrame(annual_records)
        fig,ax=plt.subplots(figsize=(9,4))
        for key,part in frame.groupby("concept"):
            ax.plot(part["year"],part["A_tcav"],alpha=.35)
        ax.set(xlabel="Calendar year",ylabel="Median per-fit TCAV in original A",title="Frozen directions; overlapping concepts")
        display(fig); plt.close(fig)
    print("Aggregate tables shown; private fitted objects and per-fit trajectories remain on Fiji")

,A_n,A_tcav,P_A_given_R,R_n,R_tcav,activation,attempted_fits,concept,f2,family,lag,n,reference,valid_fits,variants,year
0,21,1.0,0.633333,30,1.0,0.456357,50,"r1/2007/2007/42/42:94,135:3,215:146",0.833333,"2007/42/42:94,135:3,215:146",0,237,2007,50,"last_3,all_history,baseline",2007
1,0,NaN,0.000000,34,1.0,0.000000,50,"r1/2007/2007/42/42:94,135:3,215:146",0.000000,"2007/42/42:94,135:3,215:146",1,282,2007,50,"last_3,all_history,baseline",2008
2,0,NaN,0.000000,40,1.0,0.000000,50,"r1/2007/2007/42/42:94,135:3,215:146",0.000000,"2007/42/42:94,135:3,215:146",2,296,2007,50,"last_3,all_history,baseline",2009
3,0,NaN,0.000000,40,1.0,0.000000,50,"r1/2007/2007/42/42:94,135:3,215:146",0.000000,"2007/42/42:94,135:3,215:146",3,323,2007,50,"last_3,all_history,baseline",2010
4,0,NaN,0.000000,46,1.0,0.000000,50,"r1/2007/2007/42/42:94,135:3,215:146",0.000000,"2007/42/42:94,135:3,215:146",4,319,2007,50,"last_3,all_history,baseline",2011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
728,0,NaN,0.000000,20,0.0,0.000000,50,"r1/2013/2013/42/42:212,135:276,215:238",0.000000,"2013/42/42:212,135:276,215:238",1,324,2013,50,baseline,2014
729,0,NaN,0.000000,17,0.0,0.000000,50,"r1/2013/2013/42/42:212,135:276,215:238",0.000000,"2013/42/42:212,135:276,215:238",2,293,2013,50,baseline,2015
730,47,1.0,0.777778,18,1.0,0.287014,50,"r1/2013/2013/42/42:273,135:220,215:35",0.339806,"2013/42/42:273,135:220,215:35",0,322,2013,50,baseline,2013
731,197,1.0,1.000000,21,1.0,0.815943,50,"r1/2013/2013/42/42:273,135:220,215:35",0.129790,"2013/42/42:273,135:220,215:35",1,324,2013,50,baseline,2014


,brier,brier_change,brier_high,brier_low,concept,deaths,difference_of_medians,draw,high,low,...,source,start,status,subgroup,survivors,undefined_fraction,valid_fits,value,year,zero_norm_rows
0,NaN,NaN,NaN,NaN,"r1/2007/2007/42/42:94,135:3,215:146",NaN,NaN,NaN,27.525000,11.000000,...,annual_binary,NaN,NaN,NaN,NaN,0.0,NaN,19.000000,2007,NaN
1,NaN,NaN,NaN,NaN,"r1/2007/2007/42/42:94,135:3,215:146",NaN,NaN,NaN,17.000000,6.000000,...,annual_binary,NaN,NaN,NaN,NaN,0.0,NaN,11.000000,2007,NaN
2,NaN,NaN,NaN,NaN,"r1/2007/2007/42/42:94,135:3,215:146",NaN,NaN,NaN,5.000000,0.000000,...,annual_binary,NaN,NaN,NaN,NaN,0.0,NaN,2.000000,2007,NaN
3,NaN,NaN,NaN,NaN,"r1/2007/2007/42/42:94,135:3,215:146",NaN,NaN,NaN,0.785714,0.442729,...,annual_binary,NaN,NaN,NaN,NaN,0.0,NaN,0.633333,2007,NaN
4,NaN,NaN,NaN,NaN,"r1/2007/2007/42/42:94,135:3,215:146",NaN,NaN,NaN,1.000000,0.777778,...,annual_binary,NaN,NaN,NaN,NaN,0.0,NaN,0.904762,2007,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240765,NaN,NaN,NaN,NaN,"r1/2013/2013/42/42:273,135:220,215:35",NaN,NaN,NaN,NaN,NaN,...,composition,2014.0,NaN,NaN,NaN,NaN,NaN,1.014500,2015,NaN
240766,NaN,NaN,NaN,NaN,"r1/2013/2013/42/42:273,135:220,215:35",NaN,NaN,NaN,NaN,NaN,...,composition,2014.0,NaN,NaN,NaN,NaN,NaN,0.826620,2015,NaN
240767,NaN,NaN,NaN,NaN,"r1/2013/2013/42/42:273,135:220,215:35",NaN,NaN,NaN,NaN,NaN,...,composition,2014.0,NaN,NaN,NaN,NaN,NaN,1.010780,2015,NaN
240768,NaN,NaN,NaN,NaN,"r1/2013/2013/42/42:273,135:220,215:35",NaN,NaN,NaN,NaN,NaN,...,composition,2014.0,NaN,NaN,NaN,NaN,NaN,1.049702,2015,NaN


,concept,end,entrants,first_observed_entrants,lag,nonreturners,paired_activation_change,previously_observed_entrants,returners,start
0,"r1/2007/2007/42/42:94,135:3,215:146",2008,90,90,1,45,-0.467970,0,192,2007
1,"r1/2007/2007/42/42:94,135:3,215:146",2009,131,67,2,72,-0.457135,64,165,2007
2,"r1/2007/2007/42/42:94,135:3,215:146",2010,167,67,3,81,-0.453770,100,156,2007
3,"r1/2007/2007/42/42:94,135:3,215:146",2011,221,83,4,139,-0.488848,138,98,2007
4,"r1/2007/2007/42/42:94,135:3,215:146",2012,223,48,5,149,-0.509613,175,88,2007
...,...,...,...,...,...,...,...,...,...,...
1032,"r1/2013/2013/42/42:212,135:276,215:238",2015,66,12,2,95,-0.494343,54,227,2013
1033,"r1/2013/2013/42/42:212,135:276,215:238",2015,28,12,2,59,0.000000,16,265,2014
1034,"r1/2013/2013/42/42:273,135:220,215:35",2014,64,42,1,62,0.561089,22,260,2013
1035,"r1/2013/2013/42/42:273,135:220,215:35",2015,66,12,2,95,0.775136,54,227,2013


<Figure size 900x400 with 1 Axes>

Aggregate tables shown; private fitted objects and per-fit trajectories remain on Fiji


## Checks

Synthetic checks call the actual helpers. They test fixed class mass, deterministic resampling, duplicate multiplicities, orthogonal signs, common-draw width comparisons, fallback selection, exact pairing, mixture reconciliation, zero removal and independent numerical derivatives. Check mode loads no real patient data.

In [22]:
CHECK_ROWS=[]
print('Synthetic checks: actual analysis helpers; no patient inputs')

Synthetic checks: actual analysis helpers; no patient inputs


In [23]:
y=np.array([1,1,0,0,0])
mult=np.array([2,1,1,2,0])
w=class_weights(y,mult,4)
assert np.isclose(w[y==1].sum(),2) and np.isclose(w[y==0].sum(),2) and w[-1]==0
CHECK_ROWS.append({'check':"fixed total/class mass and absent-patient weights",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,fixed total/class mass and absent-patient weights,passed


In [24]:
a=draw_multiplicity(y,1001,"legacy")
b=draw_multiplicity(y,1001,"all_negatives")
np.testing.assert_array_equal(a,draw_multiplicity(y,1001,"legacy"))
assert a[y==1].sum()==a[y==0].sum()==2 and b[y==1].sum()==2 and b[y==0].sum()==3
CHECK_ROWS.append({'check':"deterministic paired stratified sampling",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,deterministic paired stratified sampling,passed


In [25]:
X=np.array([[-2.,0],[-1.,0],[1.,0],[2.,0]])
fit=fit_cav(X,np.array([0,0,1,1]),np.ones(4),4,.1,np.array(list("abcd")))
duplicated=fit_cav(X,np.array([0,0,1,1]),np.array([2,1,1,2]),4,.1,np.array(list("abcd")))
expanded=np.repeat(np.arange(4),[2,1,1,2])
equivalent=fit_cav(X[expanded],np.array([0,0,1,1])[expanded],np.ones(6),4,.1,np.array(list("abcd"))[expanded])
np.testing.assert_allclose(duplicated["model"].coef_,equivalent["model"].coef_,rtol=1e-5,atol=1e-7)
assert fit["status"]=="measured" and fit_cav(X,np.array([0,0,1,1]),np.array([1,0,1,1]),4,.1,np.array(list("abcd")))["status"]=="unsupported"
CHECK_ROWS.append({'check':"bootstrap multiplicity equals repeated-observation weighting; distinct support",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,bootstrap multiplicity equals repeated-observa...,passed


In [26]:
projections,ids=projected_metrics([{"status":"measured","direction":np.array([1.,0.])},{"status":"measured","direction":np.array([1.,1e-4])/np.linalg.norm([1.,1e-4])}],np.array([[1e-6,-1.],[-1e-6,1.]]),"cpu")
assert (projections[:,0]>0).tolist()!= (projections[:,1]>0).tolist()
CHECK_ROWS.append({'check':"near-orthogonal gradients separate sign from cosine",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,near-orthogonal gradients separate sign from c...,passed


In [27]:
fallback=select_procedure({})
assert fallback["setting"]==("legacy",.1) and fallback["status"]=="not_computable"
CHECK_ROWS.append({'check':"uncomputable noise/width selection retains legacy",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,uncomputable noise/width selection retains legacy,passed


In [28]:
unit={"patients":np.array(["a","b","a","c","d","a","c"]),"years":np.array([2007,2007,2008,2008,2008,2010,2010]),
          "roles":np.array(["evaluation"]*7),"activation":np.array([1.,3.,2.,4.,8.,5.,7.]),"rule_mask":np.array([1,0,1,0,1,1,1],bool),
          "first_observed_year":np.array([2007,2007,2007,2006,2008,2007,2006]),"cutoff":2.}
weights=np.ones((3,7))
change=composition(unit,2007,2008,weights,np.ones((7,2)))
assert change["returners"]==1 and change["entrants"]==2 and change["nonreturners"]==1
assert change["first_observed_entrants"]==1 and change["previously_observed_entrants"]==1
assert change["paired_activation_change"]==1 and sum(change["transitions"]["A"].values())==1
assert np.isclose((change["returners"]*change["returner_end_mean"]+change["entrants"]*change["entrant_mean"])/change["end_n"],change["end_mean"])
assert composition(unit,2009,2010,weights,np.ones((7,2)))["returners"]==0
CHECK_ROWS.append({'check':"pair-first transitions, reappearance entrants, mixture identity, no missing-year bridge",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,"pair-first transitions, reappearance entrants,...",passed


In [29]:
h=np.arange(12.).reshape(3,4)
activation=np.array([0.,1.,2.])
decoder=np.array([1.,0.,0.,0.])
scale=np.array([2.,1.,1.,1.])
removed,control,component=removal_embeddings(h,activation,decoder,scale)
np.testing.assert_array_equal(removed+component,h)
np.testing.assert_allclose(np.linalg.norm(control-h,axis=1),np.linalg.norm(component,axis=1))
zero,_,_=removal_embeddings(h,np.zeros(3),decoder,scale)
np.testing.assert_array_equal(zero,h)
CHECK_ROWS.append({'check':"zero-removal identity, residual/bias preservation, norm-matched control",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,"zero-removal identity, residual/bias preservat...",passed


In [30]:
metrics=binary_metrics([],[])
assert np.isnan(metrics["f2"]) and interval([np.nan])["undefined_fraction"]==1
CHECK_ROWS.append({'check':"empty/undefined support remains undefined",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,empty/undefined support remains undefined,passed


In [31]:
yy=np.array([0,1,0,1])
scores=np.array([.2,.5,.5,.9])
ww=np.array([[1,1,1,1],[2,1,3,2],[0,1,0,1]])
auc=weighted_auc(yy,scores,ww)
for i in range(2):
        assert np.isclose(auc[i],roc_auc_score(yy,scores,sample_weight=ww[i]))
assert np.isnan(auc[2])
CHECK_ROWS.append({'check':"weighted AUROC matches sklearn including score ties and absent class",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,weighted AUROC matches sklearn including score...,passed


In [32]:
from types import SimpleNamespace
head=torch.nn.Linear(2,2,bias=True).double()
with torch.no_grad():
        head.weight.copy_(torch.tensor([[0.,0.],[.3,-.2]],dtype=torch.float64)); head.bias.zero_()
continuation=SimpleNamespace(head=head,temperature=0.,configuration=None,reweight=False,eval_ys=None,
        parse=lambda logits,configuration:logits[:,0,:],aggregate=lambda logits,reweight,labels:torch.softmax(logits,dim=-1))
xx=torch.tensor([[.2,.3],[1.,-.2]],dtype=torch.float64,requires_grad=True)
probability=double_probability(continuation,xx)[:,1]
derivative,=torch.autograd.grad(probability.sum(),xx)
oracle=probability.detach().numpy()[:,None]*(1-probability.detach().numpy()[:,None])*np.array([.3,-.2])
np.testing.assert_allclose(derivative.detach().numpy(),oracle,rtol=1e-12,atol=1e-12)
step=torch.tensor([[1.,0.],[0.,1.]],dtype=torch.float64)
fd=(double_probability(continuation,xx+.001*step)[:,1]-double_probability(continuation,xx-.001*step)[:,1])/.002
assert torch.all(torch.abs((derivative*step).sum(1)-fd)<=1e-8+1e-4*torch.abs(fd))
CHECK_ROWS.append({'check':"independent double head derivative and finite differences",'result':'passed'})
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,independent double head derivative and finite ...,passed


In [33]:
def selection_checks():
    settings=[(m,c) for m in METHODS for c in C_GRID]
    scores={}
    for setting in settings:
        baseline=setting==("legacy",.1)
        scores[setting]={"fit_ids":[0,1],"discrimination":[{"auroc":.8 if baseline else .79}]*2,
                        "strata":{"A":({"n":10},{"tcav":{"per_fit":np.array([0.,1.]) if baseline else np.array([.4,.6])}}),
                                  "R":({"n":0},{})}}
    result={"aliases":["last_3"],"reference":2007,"status":"measured","scores":scores,
            "negative_scores":{("legacy",.1):[.7,.8,.75]}}
    selected=select_procedure({"fixture":result})
    assert selected["setting"]==("all_negatives",.1) and selected["strata"]==["A"]
    result["negative_scores"][("legacy",.1)]=[.7,None,.8]
    assert select_procedure({"fixture":result})["setting"]==("legacy",.1)
    return ["one-stratum adoption, deterministic ties and incomplete-noise fallback"]
def empty_stratum_checks():
    scores={"strata":{"A":({"status":"undefined","n":0,"fits":1},{})},"fit_ids":[0],"cosines":{}}
    rows=reference_strata_rows({"concept":"fixture"},scores,0)
    assert rows[0]["n"]==0 and rows[0]["W"] is None and "tcav" not in rows[0]
    return ["empty stratum keeps its ledger row without per-fit values"]
CHECK_ROWS.extend({'check':label,'result':'passed'} for label in selection_checks()+empty_stratum_checks())
display(pd.DataFrame(CHECK_ROWS[-2:]))

,check,result
0,"one-stratum adoption, deterministic ties and i...",passed
1,empty stratum keeps its ledger row without per...,passed


In [34]:
def development_poison_check():
    from unittest.mock import patch
    rng=np.random.default_rng(9)
    n=18; rule=np.arange(n)%2==0
    X=np.c_[np.where(rule,2.,-2.),rng.normal(size=n)]
    unit={"embeddings":X,"activation":rule.astype(float)*2,"rule_mask":rule,"patients":np.array([str(i) for i in range(n)]),
          "cav_fit":np.arange(8),"selection":np.arange(8,16),"cutoff":1.,"gradients":rng.normal(size=(n,2)),
          "identity":{"fixture":True},"aliases":["last_3"],"reference":2007}
    poisoned={k:(v.copy() if isinstance(v,np.ndarray) else deepcopy(v)) for k,v in unit.items()}
    for key in ("embeddings","activation","gradients"):
        poisoned[key][16:]=np.nan
    run_jobs=lambda **kwargs:lambda jobs:[function(*args,**kw) for function,args,kw in jobs]
    with patch.dict(globals(),{"Parallel":run_jobs}):
        before=assess_reference(unit,"cpu",(1001,1002))
        after=assess_reference(poisoned,"cpu",(1001,1002))
    for setting in before["joint"]:
        for left,right in zip(before["joint"][setting],after["joint"][setting]):
            assert left["status"]==right["status"]
            if left["status"]=="measured":
                np.testing.assert_array_equal(left["direction"],right["direction"])
    assert before["negative_cutoff"]==after["negative_cutoff"]==0
    return ["evaluation/future poisoning cannot alter reference CAV fits or cutoff"]
CHECK_ROWS.extend({'check':label,'result':'passed'} for label in development_poison_check())
display(pd.DataFrame(CHECK_ROWS[-1:]))

,check,result
0,evaluation/future poisoning cannot alter refer...,passed


In [35]:
fixture_unit={"key":"synthetic","reference":2007,"years":np.array([2007,2007,2015,2015]),
    "roles":np.array(["evaluation"]*4),"activation":np.array([1.,0.,1.,0.]),"cutoff":.5,
    "rule_mask":np.array([True,True,True,False]),"gradients":np.array([[0.,0.],[1.,0.],[0.,0.],[1.,0.]]),
    "labels":np.array([1.,0.,np.nan,np.nan]),"probabilities":np.array([[.2,.8],[.7,.3],[.3,.7],[.6,.4]])}
fixture_weights=np.ones((4,4)); fixture_sensitivity=fixture_unit["gradients"][:,:1]
fixture_annual={}
for fixture_year in (2007,2015):
    summary,details=annual_summary(fixture_unit,[],fixture_year,fixture_weights,fixture_sensitivity)
    summary["yearly_description"]={"status":"missing_dependency"}
    fixture_annual[fixture_year]={"summary":summary,"per_fit":details,"fit_seeds":[1001]}
fixture_records=evaluation_tables(fixture_unit,fixture_annual,[])
fixture_support=[row for row in fixture_records if row["source"]=="annual_subgroup_support"]
assert len(fixture_support)==4, "annual subgroup mortality and zero-gradient support must be public"
fixture_R=next(row for row in fixture_support if row["year"]==2007 and row["subgroup"]=="R")
assert fixture_R["n"]==2 and fixture_R["deaths"]==1 and fixture_R["survivors"]==1
assert fixture_R["zero_norm_rows"]==1 and fixture_R["norm_quantity"]=="raw_probability_gradient"
assert np.isclose(fixture_R["brier"],.065) and fixture_R["mortality"]==.5
assert np.isclose(fixture_R["brier_low"],.065) and np.isclose(fixture_R["brier_high"],.065)
assert fixture_R["mortality_low"]==fixture_R["mortality_high"]==.5
assert all(row["outcome_status"]=="undefined" and row["deaths"] is None and row["brier"] is None
           for row in fixture_support if row["year"]==2015)
from cmr_common import public_records
public_records(clean_records(fixture_support),set().union(*(row.keys() for row in fixture_support)))
CHECK_ROWS.append({"check":"public annual mortality, zero-gradient support, conditional intervals and 2015 exclusion","result":"passed"})
display(pd.DataFrame(CHECK_ROWS[-1:]))


,check,result
0,"public annual mortality, zero-gradient support...",passed


In [36]:
CHECKS=pd.DataFrame(CHECK_ROWS)
display(CHECKS)
assert (CHECKS['result']=='passed').all()
if MODE!='check':
    finish_run(ctx,'completed',{'families':len(reference_results) if MODE!='evaluate' else len(private_evaluation),'checks':len(CHECKS)})

,check,result
0,fixed total/class mass and absent-patient weights,passed
1,deterministic paired stratified sampling,passed
2,bootstrap multiplicity equals repeated-observa...,passed
3,near-orthogonal gradients separate sign from c...,passed
4,uncomputable noise/width selection retains legacy,passed
5,"pair-first transitions, reappearance entrants,...",passed
6,"zero-removal identity, residual/bias preservat...",passed
7,empty/undefined support remains undefined,passed
8,weighted AUROC matches sklearn including score...,passed
9,independent double head derivative and finite ...,passed
